# 🎮 PokemonAgent Training on Google Colab

Train your MaskablePPO Pokémon agent directly on Google Colab with a native Pokémon Showdown server!

## Cell 1️⃣: Install Node.js

In [ ]:
# Install Node.js (required for Pokémon Showdown server)
!curl -fsSL https://deb.nodesource.com/setup_18.x | sudo -E bash -
!apt-get install -y nodejs
!node --version
!npm --version
print("✅ Node.js installed!")

## Cell 2️⃣: Clone Repository & Install Python Dependencies

In [ ]:
import os

# Clone the repository (UPDATE THIS WITH YOUR REPO URL)
repo_url = "https://github.com/shayro9/PokemonAgent.git"
branch = "Supervised"
!cd /content && git clone -b {branch} {repo_url}
os.chdir('/content/PokemonAgent')

# Install Python dependencies
!pip install -q -r requirements.txt

print("✅ Repository and dependencies installed!")

## Cell 3️⃣: Build Pokémon Showdown Server

In [ ]:
import os

# Clone Pokémon Showdown
print("⏳ Cloning Pokémon Showdown...")
!cd /content && git clone --depth 1 https://github.com/smogon/pokemon-showdown.git
os.chdir('/content/pokemon-showdown')

# Build the server
print("⏳ Installing dependencies and building...")
!npm install -q
!node build

print("✅ Pokémon Showdown server built successfully!")

## Cell 4️⃣: Start Pokémon Showdown Server (Background)

In [ ]:
import subprocess
import time
import os

os.chdir('/content/pokemon-showdown')

# Start the server in the background
print("🚀 Starting Pokémon Showdown server on localhost:8000...")
server_process = subprocess.Popen(
    ["node", "pokemon-showdown", "start", "--no-security"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Give server time to start
time.sleep(5)

# Check if server is running
if server_process.poll() is None:
    print(f"✅ Server started successfully!")
    print(f"   Process ID: {server_process.pid}")
else:
    stdout, stderr = server_process.communicate()
    print(f"❌ Server failed to start")
    print(f"Error: {stderr}")

## Cell 5️⃣: Verify Server Connection

In [ ]:
import urllib.request
import urllib.error
import time

def test_server():
    """Test if the Pokémon Showdown server is reachable."""
    max_retries = 15
    for attempt in range(max_retries):
        try:
            response = urllib.request.urlopen("http://localhost:8000/", timeout=2)
            print(f"✅ Server is running and reachable!")
            return True
        except (urllib.error.URLError, ConnectionRefusedError) as e:
            if attempt < max_retries - 1:
                print(f"⏳ Waiting for server (attempt {attempt + 1}/{max_retries})...")
                time.sleep(2)
            else:
                print(f"❌ Cannot reach server: {e}")
                return False
    return False

test_server()

## Cell 6️⃣: Configure W&B (Optional but Recommended)

Get your API key from https://wandb.ai/settings/api

In [ ]:
import os

# Set your W&B API key (optional, but recommended for tracking)
# Get it from: https://wandb.ai/settings/api
wandb_api_key = "your-wandb-api-key-here"

if wandb_api_key != "your-wandb-api-key-here":
    os.environ["WANDB_API_KEY"] = wandb_api_key
    print("✅ W&B configured")
else:
    print("⚠️  W&B API key not set (optional)")

## Cell 7️⃣: 🎯 RUN TRAINING

Edit the command below to customize your training!

In [ ]:
import os
os.chdir('/content/PokemonAgent')

# Quick test configuration (change these parameters as needed)
!python -m training.train --n-envs=4 --timesteps=3_000_000 --rounds-per-opponent=2 --eval-every-timesteps=600_000 --eval-episodes=500 --split-generated-pool --agent-data-path .\data\meta_pool_teams_6_gen1ou_db.json --format=gen1ou --model-path=models/6v6_gen1_1M_meta_self --curriculum-config .\curriculum\examples\frozen_self.yaml

## Cell 8️⃣ (Optional): Download Trained Model

In [ ]:
import os
from google.colab import files

# Compress the trained model
os.system("cd /content/PokemonAgent && zip -r /tmp/trained_model.zip data/1v1*")

# Download to your computer
files.download("/tmp/trained_model.zip")
print("✅ Model downloaded to your computer!")

## Cell 9️⃣ (Optional): Stop Server & Cleanup

In [ ]:
# Kill the Pokémon Showdown server process
import subprocess

# Find and kill the pokemon-showdown process
result = subprocess.run(['pgrep', '-f', 'pokemon-showdown'], capture_output=True, text=True)
if result.stdout.strip():
    pids = result.stdout.strip().split('\n')
    for pid in pids:
        print(f"🛑 Killing process {pid}...")
        os.kill(int(pid), 9)
    print("✅ Server stopped")
else:
    print("ℹ️  No server process found (already stopped)")

## 📋 Training Command Presets

Replace Cell 7 with one of these for different training configurations:

### Quick Test (5 min)
```bash
python -m training.train \
  --train-team steelix \
  --pool toxapex \
  --timesteps 10000 \
  --rounds-per-opponent 100 \
  --device cuda
```

### Balanced Training (1-2 hours)
```bash
python -m training.train \
  --train-team garchomp \
  --pool-all \
  --timesteps 200000 \
  --rounds-per-opponent 2000 \
  --device cuda
```

### Full Training with Evaluation (3-5 hours)
```bash
python -m training.train \
  --train-team steelix \
  --pool-all \
  --timesteps 500000 \
  --rounds-per-opponent 2000 \
  --eval-every-timesteps 50000 \
  --eval-episodes 100 \
  --device cuda
```